# Introducción a la Inteligencia Artificial en Ingeniería de Energía – 2026
## Semana 4
### Gabriel Govea
---

## Parte 1 – Definiciones Conceptuales
---

### a) Inferencia vs. Predicción

Aunque estos términos se usan indistintamente en el lenguaje cotidiano, en estadística y aprendizaje automático tienen objetivos bien diferenciados:

| | **Predicción** | **Inferencia** |
|---|---|---|
| **Objetivo** | Estimar el valor de $Y$ con la mayor precisión posible | Entender *cómo* y *por qué* $X$ afecta a $Y$ |
| **Prioridad** | Minimizar el error de generalización | Interpretabilidad y significancia estadística |
| **Modelos típicos** | Redes Neuronales, Gradient Boosting (caja negra) | Regresión Lineal, GAM (caja blanca) |
| **Pregunta clave** | ¿Cuál será el valor futuro? | ¿Es esta relación estadísticamente significativa? |


### b) Modelo, Features, Target, Hiperparámetros

- **Modelo:** Función matemática o algoritmo que mapea entradas ($X$) a salidas ($Y$). Puede ser paramétrico (asume forma funcional previa) o no paramétrico (aprende la forma de los datos).
- **Features ($X$):** Variables independientes o entradas. La calidad del modelo depende críticamente de su ingeniería: *"Garbage In, Garbage Out"*.
- **Target ($Y$):** Variable dependiente que se desea predecir o explicar (ej. producción, facies, precio).
- **Hiperparámetros:** Configuraciones definidas *antes* del entrenamiento que controlan el proceso de aprendizaje (ej. número de clusters $k$, tasa de aprendizaje, profundidad de árbol). Se optimizan mediante validación cruzada o búsqueda en grilla, a diferencia de los *parámetros* del modelo que se aprenden automáticamente durante el entrenamiento.


### c) No linealidad, Multicolinealidad, Heterocedasticidad

| Concepto | Definición | Implicación Práctica | Solución |
|---|---|---|---|
| **No linealidad** | El cambio en $Y$ no es proporcional al cambio en $X$ | Modelos lineales presentarán alto sesgo (underfitting) | Modelos no lineales (Árboles, SVM), transformaciones de features |
| **Multicolinealidad** | Dos o más features están altamente correlacionadas entre sí | Estimaciones de coeficientes inestables; infla la varianza | Eliminar variables redundantes, PCA, regularización Ridge/Lasso |
| **Heterocedasticidad** | La varianza del error no es constante en el rango de datos | Los errores estándar y valores-p no son confiables para inferencia | Transformaciones logarítmicas, modelos robustos, WLS |


### d) Maldición de la Dimensionalidad

> **Fenómeno:** A medida que aumenta el número de dimensiones (features), el volumen del espacio crece exponencialmente y los datos se vuelven extremadamente dispersos (*sparse*).

**Consecuencias principales:**
- Las métricas de distancia (euclidiana) pierden significado: todos los puntos parecen estar a la misma distancia.
- Se requiere una cantidad de datos de entrenamiento que crece exponencialmente con las dimensiones.
- El riesgo de overfitting aumenta drásticamente.

**Soluciones:** Selección de features, reducción de dimensionalidad (PCA, t-SNE, UMAP), regularización L1/L2.


### e) Proyección, Compresión, Codificación, Espacio Latente

Aunque tienen matices técnicos distintos, todos comparten un concepto unificador: **reducción de la complejidad representacional** — mapear datos de un espacio de alta dimensión a uno de menor dimensión preservando la información esencial.

| Término | Definición | Ejemplo |
|---|---|---|
| **Proyección** | Transformación geométrica a un subespacio de menor dimensión | PCA proyecta datos sobre componentes principales ortogonales |
| **Compresión** | Reducir el tamaño de la representación minimizando pérdida de información | Autoencoders, JPEG |
| **Codificación** | Convertir datos a un formato numérico útil para el modelo | Word Embeddings, One-Hot Encoding |
| **Latente** | Variables no observadas directamente que explican la estructura de los datos | Espacio latente en VAEs, NMF, t-SNE |


---
## Parte 2 – Reducción de Dimensionalidad
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm

# Reducción de Dimensionalidad
from sklearn.decomposition import PCA, NMF, KernelPCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline

# Clustering
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn import metrics as skl_metrics
from skimage import metrics as ski_metrics

def metricas_clustering(true, pred, features=None):
    """Calcula métricas estándar de evaluación para clustering."""
    resultado = {
        'ARI':     skl_metrics.adjusted_rand_score(true, pred),
        'AMI':     skl_metrics.adjusted_mutual_info_score(true, pred),
        'V-Medida': skl_metrics.v_measure_score(true, pred),
    }
    if features is not None:
        resultado.update({
            'Silhouette': skl_metrics.silhouette_score(features, pred),
            'DBI':        skl_metrics.davies_bouldin_score(features, pred),
            'CHI':        skl_metrics.calinski_harabasz_score(features, pred),
        })
    return resultado

print('✅ Librerías cargadas correctamente.')


### 2a. Carga y exploración del dataset de facies


In [ ]:
# Carga de una realización del dataset de reservorios deepwater
# Ajustar la ruta según el entorno local
ruta_facies = 'data/facies/facies_X.npy'  # Reemplazar X por el número de realización

facies = np.load(ruta_facies)
nx, ny, nz = facies.shape

print(f'Forma del dataset: {facies.shape}')
print(f'Dimensiones: nx={nx}, ny={ny}, nz={nz}')
print(f'Total de elementos: {nx*ny*nz:,}')
print(f'Rango de valores: [{facies.min():.4f}, {facies.max():.4f}]')


### 2b. Visualización 3D con PyVista y cortes en Z con Matplotlib


In [ ]:
# --- Visualización con PyVista (plot 3D) ---
try:
    import pyvista as pv
    grid = pv.ImageData()
    grid.dimensions = [nx, ny, nz]
    grid.origin    = [0, 0, 0]
    grid.spacing   = [1, 1, 1]
    grid['facies'] = facies.flatten(order='F')
    plotter = pv.Plotter(notebook=True)
    plotter.add_mesh(grid, scalars='facies', cmap='jet', opacity=0.6)
    plotter.add_axes()
    plotter.add_title('Visualización 3D – Facies de Reservorio')
    plotter.show()
except Exception as e:
    print(f'PyVista no disponible o error: {e}')

# --- Cortes en el eje Z con matplotlib ---
indices_z = np.linspace(0, nz - 1, 9, dtype=int)
fig, ejes = plt.subplots(3, 3, figsize=(11, 11))

for ax, k in zip(ejes.flatten(), indices_z):
    im = ax.imshow(facies[:, :, k], cmap='jet')
    ax.set_title(f'Z = {k}', fontsize=10)
    ax.axis('off')

fig.colorbar(im, ax=ejes, orientation='vertical', fraction=0.02, pad=0.04)
fig.suptitle('Cortes en el eje Z – Realización de Facies', fontsize=14)
plt.tight_layout()
plt.show()


**Observaciones:** Los cortes revelan elementos arquitectónicos propios de ambientes de aguas profundas (canales, lóbulos). La distribución espacial de los valores muestra heterogeneidad tanto lateral como vertical, lo que indica la presencia de múltiples ambientes deposicionales con distintas propiedades petrofísicas.


### 2c. Base matemática del método de Reducción de Dimensionalidad


#### Análisis de Componentes Principales (PCA)

**Supuestos:**
- Las direcciones de máxima varianza contienen la información más relevante (señal vs. ruido).
- Las relaciones entre variables son lineales.
- Los componentes resultantes son ortogonales entre sí (no correlacionados).

**Ecuaciones clave:**

Dada la matriz de datos centrada $X \in \mathbb{R}^{N \times D}$:

1. Matriz de covarianza: $\quad C = \dfrac{1}{N-1} X^T X$

2. Descomposición espectral: $\quad C\, v = \lambda\, v$

3. Proyección al espacio latente: $\quad Z = X\, W_k$ &nbsp; (usando los $k$ eigenvectores con mayor $\lambda$)

4. Reconstrucción: $\quad \hat{X} = Z\, W_k^T$

**Base algorítmica:** (1) Centrar los datos. (2) Calcular $C$. (3) Obtener eigenvalores y eigenvectores. (4) Ordenar por eigenvalor descendente y seleccionar los $k$ primeros. (5) Proyectar.

**Limitación principal:** Los nuevos features $Z$ son combinaciones lineales de los originales y pierden interpretabilidad física directa.


### 2d. Aplicación de PCA con distintas dimensiones latentes


In [ ]:
# Preparar el dataset: (nx, ny, nz) → (nz, nx*ny)
facies_flat = np.moveaxis(facies, -1, 0).reshape(nz, nx * ny)
print(f'Dataset aplanado para PCA: {facies_flat.shape}  (muestras × features)')

# Estandarización (PCA es sensible a la escala)
escalador = StandardScaler()
X_norm = escalador.fit_transform(facies_flat)

# Ajustar PCA completo para estudiar la varianza explicada
t0 = time.time()
pca_completo = PCA()
pca_completo.fit(X_norm)
t_fit = time.time() - t0

print(f'Tiempo de ajuste (PCA completo): {t_fit:.3f} s')
print(f'Número máximo de componentes: {pca_completo.n_components_}')

# Curva de varianza explicada acumulada
var_acum = np.cumsum(pca_completo.explained_variance_ratio_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(var_acum) + 1), var_acum, linewidth=2)
plt.axhline(0.95, color='r', linestyle='--', label='Umbral 95%')
plt.xlabel('Número de Componentes')
plt.ylabel('Varianza Explicada Acumulada')
plt.title('Espectro de Varianza – PCA')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


### 2e. Reconstrucción y métricas de calidad de imagen


In [ ]:
def reconstruir_pca(X_norm, escalador, n_comp, forma_original):
    """Aplica PCA con n_comp componentes y reconstruye al espacio original."""
    pca = PCA(n_components=n_comp)
    Z   = pca.fit_transform(X_norm)
    Xhat_norm = pca.inverse_transform(Z)
    Xhat = escalador.inverse_transform(Xhat_norm)
    nz, nx, ny = forma_original
    return Xhat.reshape(nz, nx, ny).transpose(1, 2, 0)  # → (nx, ny, nz)

# Reconstrucción con n=20 componentes como ejemplo inicial
forma = (nz, nx, ny)
facies_rec = reconstruir_pca(X_norm, escalador, 20, forma)

drange = facies.max() - facies.min()
mse_val  = ski_metrics.mean_squared_error(facies, facies_rec)
ssim_val = ski_metrics.structural_similarity(facies, facies_rec, data_range=drange, channel_axis=2)
psnr_val = ski_metrics.peak_signal_noise_ratio(facies, facies_rec, data_range=drange)
print(f'n=20  →  MSE: {mse_val:.4f} | SSIM: {ssim_val:.4f} | PSNR: {psnr_val:.2f} dB')

# Visualización comparativa: Original vs Reconstruido vs Error
slices_viz = [0, 20, 40, 60, 80, 100, 120]
fig, ejes = plt.subplots(3, len(slices_viz), figsize=(14, 5), sharex=True, sharey=True)

for col, k in enumerate(slices_viz):
    original    = facies[:, :, k]
    reconstruido = facies_rec[:, :, k]
    error       = np.abs((original - reconstruido) / (original + 1e-6))

    ejes[0, col].imshow(original,    cmap='jet'); ejes[0, col].set_title(f'z={k}', fontsize=8)
    ejes[1, col].imshow(reconstruido, cmap='jet')
    ejes[2, col].imshow(error,        cmap='binary')

for fila, etiq in enumerate(['Original', 'Reconstruido', 'Error Rel.']):
    ejes[fila, 0].set_ylabel(etiq, fontsize=9)

plt.suptitle('Comparación Original vs Reconstruido (n=20)', fontsize=12)
plt.tight_layout()
plt.show()


### 2f. Elbow plot – Precisión de reconstrucción vs dimensión latente


In [ ]:
dims_latentes = np.linspace(2, nz, 20, dtype=int)
metricas_rec = {'mse': [], 'ssim': [], 'psnr': []}
tiempos = []

drange = facies.max() - facies.min()

for n in tqdm(dims_latentes, desc='Evaluando dimensiones latentes'):
    t0 = time.time()
    rec = reconstruir_pca(X_norm, escalador, n, forma)
    tiempos.append(time.time() - t0)

    metricas_rec['mse'].append(ski_metrics.mean_squared_error(facies, rec))
    metricas_rec['ssim'].append(
        ski_metrics.structural_similarity(facies, rec, data_range=drange, channel_axis=2))
    metricas_rec['psnr'].append(
        ski_metrics.peak_signal_noise_ratio(facies, rec, data_range=drange))

# --- Elbow Plot ---
fig, ax1 = plt.subplots(figsize=(9, 5))

ax1.plot(dims_latentes, metricas_rec['ssim'], marker='s', label='SSIM', color='tab:blue')
ax1.set_xlabel('Dimensión Latente')
ax1.set_ylabel('SSIM')
ax1.set_ylim(0, 1.05)

# Marcar el punto de intersección al 95% de SSIM
ssim_arr = np.array(metricas_rec['ssim'])
idx_95 = np.where(ssim_arr >= 0.95)[0]
if len(idx_95) > 0:
    dim_95 = dims_latentes[idx_95[0]]
    ax1.axhline(0.95,  linestyle='--', color='grey', alpha=0.8, label='SSIM = 95%')
    ax1.axvline(dim_95, linestyle='--', color='red',  alpha=0.8, label=f'Dim = {dim_95}')
    print(f'✅ SSIM ≥ 95% alcanzado con {dim_95} dimensiones latentes')
else:
    print('⚠️  SSIM no alcanza 95% en el rango explorado')

ax2 = ax1.twinx()
ax2.plot(dims_latentes, metricas_rec['psnr'], marker='^', color='tab:green', label='PSNR')
ax2.set_ylabel('PSNR (dB)')

lineas  = ax1.get_legend_handles_labels()
lineas2 = ax2.get_legend_handles_labels()
ax1.legend(lineas[0] + lineas2[0], lineas[1] + lineas2[1], loc='lower right')
ax1.grid(True, alpha=0.4)
plt.title('Elbow Plot – Precisión de Reconstrucción vs Dimensión Latente (PCA)')
plt.tight_layout()
plt.show()

print(f'Tiempo promedio por iteración: {np.mean(tiempos):.3f} s')


### 2g. Comparación entre realizaciones geológicas distintas


In [ ]:
# Cargar una segunda realización de un ambiente geológico diferente
# Cambiar las rutas según las realizaciones disponibles
rutas_realizaciones = [
    'data/facies/facies_X.npy',   # Realización 1 (ya cargada)
    'data/facies/facies_Y.npy',   # Realización 2 – ambiente distinto
]

resultados_por_realizacion = []

for ruta in rutas_realizaciones:
    try:
        vol = np.load(ruta)
        nz_r, nx_r, ny_r = vol.shape[2], vol.shape[0], vol.shape[1]
        flat = np.moveaxis(vol, -1, 0).reshape(nz_r, nx_r * ny_r)
        esc  = StandardScaler()
        Xn   = esc.fit_transform(flat)
        dr   = vol.max() - vol.min()

        ssim_vals = []
        dims_test = np.linspace(2, nz_r, 15, dtype=int)
        for n in dims_test:
            rec = reconstruir_pca(Xn, esc, n, (nz_r, nx_r, ny_r))
            ssim_vals.append(
                ski_metrics.structural_similarity(vol, rec, data_range=dr, channel_axis=2))

        resultados_por_realizacion.append((ruta.split('/')[-1], dims_test, ssim_vals))
        print(f'{ruta.split("/")[-1]}: procesado correctamente')
    except FileNotFoundError:
        print(f'Archivo no encontrado: {ruta} — omitido')

# Comparar elbow plots de ambas realizaciones
if resultados_por_realizacion:
    plt.figure(figsize=(8, 4))
    for nombre, dims, ssim_v in resultados_por_realizacion:
        plt.plot(dims, ssim_v, marker='o', label=nombre)
    plt.axhline(0.95, linestyle='--', color='grey', alpha=0.7, label='SSIM = 95%')
    plt.xlabel('Dimensión Latente')
    plt.ylabel('SSIM')
    plt.title('Comparación entre Realizaciones – SSIM vs Dimensión Latente')
    plt.legend()
    plt.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.show()


---
## Parte 3 – Clustering
---

### 3a–b. Carga y limpieza del dataset de muestras de núcleo


In [ ]:
# Cargar el dataset de porosidad vs permeabilidad
# Disponibles: nonlinear_core_v1.csv, v2.csv, v3.csv
df = pd.read_csv('data/poro_perm/nonlinear_core_v1.csv')

print(f'Forma inicial: {df.shape}')
print(f'Columnas: {list(df.columns)}')
print(f'\nValores nulos por columna:')
print(df.isnull().sum())
print(f'\nEstadísticas descriptivas:')
df.describe()


In [ ]:
# --- Ingeniería de Features ---
# 1. Remover valores nulos
df_limpio = df.dropna().copy()

# 2. Detección y eliminación de outliers mediante Z-score
from scipy import stats
features_cols = [c for c in df_limpio.columns if c != 'CLASS']
mascara_outliers = (np.abs(stats.zscore(df_limpio[features_cols])) < 3).all(axis=1)
df_limpio = df_limpio[mascara_outliers]
print(f'Registros tras limpieza: {len(df_limpio)} (removidos: {len(df) - len(df_limpio)})')

# 3. Separar features y etiquetas verdaderas
features  = df_limpio[features_cols]
target    = df_limpio['CLASS']

# 4. Normalización (crítica para algoritmos basados en distancias)
escalador_clust = MinMaxScaler()
features_norm   = pd.DataFrame(
    escalador_clust.fit_transform(features), columns=features_cols)

# 5. Visualización del dataset limpio
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))

ejes[0].scatter(features.iloc[:, 0], features.iloc[:, 1],
                c=target, cmap='tab10', alpha=0.6)
ejes[0].set(xlabel=features_cols[0], ylabel=features_cols[1],
            title='Dataset Limpio – Escala Original')
ejes[0].grid(True, alpha=0.3)

ejes[1].scatter(features_norm.iloc[:, 0], features_norm.iloc[:, 1],
                c=target, cmap='tab10', alpha=0.6)
ejes[1].set(xlabel=features_cols[0], ylabel=features_cols[1],
            title='Dataset Limpio – Normalizado')
ejes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 3c. Base matemática del algoritmo de Clustering

#### K-Means

**Supuestos:** Clusters convexos, isótropos (esféricos), con varianza similar entre grupos y tamaño comparable.

**Función objetivo (Within-Cluster Sum of Squares):**

$$J = \sum_{i=1}^{k} \sum_{x \in C_i} \|x - \mu_i\|^2$$

donde $\mu_i$ es el centroide del cluster $C_i$.

**Algoritmo:**
1. Inicializar $k$ centroides (aleatoriamente o con *k-means++*).
2. Asignar cada punto al centroide más cercano (distancia euclidiana).
3. Recalcular centroides como la media de los puntos asignados.
4. Repetir 2–3 hasta convergencia (cambio en $J < \epsilon$).

**Limitación clave:** Falla en clusters no convexos o de forma irregular (donde DBSCAN o GMM son preferibles).


### 3d–e. Aplicación de K-Means y métricas de evaluación


In [ ]:
# Ajuste inicial con K-Means (k=3 como punto de partida)
t0 = time.time()
kmeans_base = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_base.fit(features_norm)
etiquetas_base = kmeans_base.labels_
print(f'Tiempo de ajuste (K-Means base): {time.time()-t0:.3f} s')

# Métricas de evaluación
m_base = metricas_clustering(target, etiquetas_base, features=features_norm)
print('\nMétricas – K-Means base (k=3):')
for k, v in m_base.items():
    print(f'  {k}: {v:.4f}')

# Visualización inicial
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
colores = target.values
plt.scatter(features_norm.iloc[:, 0], features_norm.iloc[:, 1],
            c=colores, cmap='tab10', alpha=0.6)
plt.title('Etiquetas Reales')
plt.xlabel(features_cols[0]); plt.ylabel(features_cols[1])
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(features_norm.iloc[:, 0], features_norm.iloc[:, 1],
            c=etiquetas_base, cmap='tab10', alpha=0.6)
plt.title('K-Means – Clusters Predichos (k=3)')
plt.xlabel(features_cols[0]); plt.ylabel(features_cols[1])
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 3f. Ajuste de hiperparámetros


In [ ]:
# Grid search manual para K-Means
param_grid = {
    'n_clusters': [2, 3, 4, 5],
    'init':       ['k-means++', 'random'],
    'max_iter':   [100, 200, 300],
    'algorithm':  ['elkan', 'lloyd'],
}

mejor_score  = -1
mejores_params = {}

t0 = time.time()
for k in param_grid['n_clusters']:
    for init in param_grid['init']:
        for max_iter in param_grid['max_iter']:
            for algo in param_grid['algorithm']:
                modelo = KMeans(n_clusters=k, init=init, max_iter=max_iter,
                                algorithm=algo, random_state=42, n_init=5)
                modelo.fit(features_norm)
                score = skl_metrics.adjusted_mutual_info_score(target, modelo.labels_)
                if score > mejor_score:
                    mejor_score   = score
                    mejores_params = dict(n_clusters=k, init=init,
                                         max_iter=max_iter, algorithm=algo)

print(f'Tiempo total de búsqueda: {time.time()-t0:.2f} s')
print(f'Mejores hiperparámetros: {mejores_params}')
print(f'Mejor AMI: {mejor_score:.4f}')

# Elbow plot para elegir k óptimo (inertia)
inercias = []
rango_k  = range(1, 10)
for k in rango_k:
    inercias.append(KMeans(n_clusters=k, random_state=42, n_init=10).fit(features_norm).inertia_)

plt.figure(figsize=(6, 3))
plt.plot(list(rango_k), inercias, marker='o')
plt.xlabel('Número de Clusters'); plt.ylabel('Inercia')
plt.title('Elbow Plot – Selección de k óptimo')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


### 3g. Ajuste final con hiperparámetros optimizados


In [ ]:
# Ajuste final con los mejores parámetros encontrados
kmeans_opt = KMeans(**mejores_params, random_state=42, n_init=10)
kmeans_opt.fit(features_norm)
etiquetas_opt = kmeans_opt.labels_

# Métricas finales
m_opt = metricas_clustering(target, etiquetas_opt, features=features_norm)
print('Métricas – K-Means optimizado:')
for k, v in m_opt.items():
    print(f'  {k}: {v:.4f}')

# Visualización final con features en escala ORIGINAL (sin normalizar)
plt.figure(figsize=(7, 5))
for lbl in np.unique(etiquetas_opt):
    mask = etiquetas_opt == lbl
    plt.scatter(features.iloc[:, 0][mask], features.iloc[:, 1][mask],
                label=f'Cluster {lbl}', alpha=0.6)
plt.xlabel(features_cols[0])
plt.ylabel(features_cols[1])
plt.title('Clustering Final Optimizado – Escala Original')
plt.legend(edgecolor='k', facecolor='lightyellow')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


### BONUS – Dataset real de laboratorio (`core.csv`)


In [ ]:
# Carga del dataset real de laboratorio
try:
    df_core = pd.read_csv('data/core.csv').dropna()
    print(f'Columnas disponibles: {list(df_core.columns)}')
    print(f'Registros: {len(df_core)}')

    # --- Clustering 2D: Porosidad vs Permeabilidad ---
    X2d = df_core[['PORO', 'PERM']].copy()
    # Transformar permeabilidad a escala logarítmica (distribución log-normal)
    X2d['PERM'] = np.log10(X2d['PERM'].clip(lower=1e-5))
    X2d_norm = StandardScaler().fit_transform(X2d)

    # Determinar k óptimo con silhouette
    scores_sil = []
    rango_k = range(2, 7)
    for k in rango_k:
        lbl = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X2d_norm)
        scores_sil.append(skl_metrics.silhouette_score(X2d_norm, lbl))

    k_opt_bonus = list(rango_k)[np.argmax(scores_sil)]
    print(f'k óptimo (Silhouette): {k_opt_bonus}')

    lbl_2d = KMeans(n_clusters=k_opt_bonus, random_state=42, n_init=10).fit_predict(X2d_norm)

    # --- Clustering 3D: Porosidad, BVO, Permeabilidad ---
    cols_3d = ['PORO', 'BVO', 'PERM']
    if all(c in df_core.columns for c in cols_3d):
        X3d = df_core[cols_3d].copy()
        X3d['PERM'] = np.log10(X3d['PERM'].clip(lower=1e-5))
        X3d_norm = StandardScaler().fit_transform(X3d)
        lbl_3d = KMeans(n_clusters=k_opt_bonus, random_state=42, n_init=10).fit_predict(X3d_norm)
    else:
        print('Columna BVO no disponible; omitiendo clustering 3D')
        lbl_3d = None

    # Visualización
    fig = plt.figure(figsize=(12, 5))

    ax1 = fig.add_subplot(121)
    ax1.scatter(df_core['PORO'], np.log10(df_core['PERM'].clip(lower=1e-5)),
                c=lbl_2d, cmap='Set1', alpha=0.7)
    ax1.set_xlabel('Porosidad'); ax1.set_ylabel('log₁₀(Permeabilidad)')
    ax1.set_title(f'Clustering 2D (k={k_opt_bonus})')
    ax1.grid(True, alpha=0.3)

    if lbl_3d is not None:
        ax2 = fig.add_subplot(122, projection='3d')
        ax2.scatter(df_core['PORO'], df_core['BVO'],
                    np.log10(df_core['PERM'].clip(lower=1e-5)),
                    c=lbl_3d, cmap='Set1', alpha=0.7)
        ax2.set_xlabel('PORO'); ax2.set_ylabel('BVO')
        ax2.set_zlabel('log₁₀(PERM)')
        ax2.set_title(f'Clustering 3D (k={k_opt_bonus})')

    plt.tight_layout()
    plt.show()

    print('El clustering sobre datos reales permite identificar unidades de flujo hidráulico')
    print('(rock types). La escala log en permeabilidad es esencial dada su distribución log-normal.')

except FileNotFoundError:
    print('Archivo core.csv no encontrado. Verifica la ruta: data/core.csv')


---
## Fin del Notebook
